In [6]:
import shap
import spacetimeformer as stf
import sys
sys.path.append('../../bats_transformer')
from data.bats_dataset import *
from tqdm import tqdm
import numpy as np
import pandas as pd

In [7]:
ignore_cols = ["FreqLedge","AmpK@end", "Fc", "FBak15dB  ", "FBak32dB", "EndF", "FBak20dB", "LowFreq", "Bndw20dB", 
               "CallsPerSec", "EndSlope", "SteepestSlope", "StartSlope", "Bndw15dB", "HiFtoUpprKnSlp", "HiFtoKnSlope", 
               "DominantSlope", "Bndw5dB", "PreFc500", "PreFc1000", "PreFc3000", "KneeToFcSlope", "TotalSlope", 
               "PreFc250", "CallDuration", "CummNmlzdSlp", "DurOf32dB", "SlopeAtFc", "LdgToFcSlp", "DurOf20dB", "DurOf15dB", 
               "TimeFromMaxToFc", "KnToFcDur", "HiFtoFcExpAmp", "AmpKurtosis", "LowestSlope", "KnToFcDmp", "HiFtoKnExpAmp", 
               "DurOf5dB", "KnToFcExpAmp", "RelPwr3rdTo1st", "LnExpB_StartAmp", "Filter", "HiFtoKnDmp", "LnExpB_EndAmp", 
               "HiFtoFcDmp", "AmpSkew", "LedgeDuration", "KneeToFcResidue", "PreFc3000Residue", "AmpGausR2", "PreFc1000Residue", 
               "Amp1stMean", "LdgToFcExp", "FcMinusEndF", "Amp4thMean", "HiFtoUpprKnExp", "HiFtoKnExp", "KnToFcExp", "UpprKnToKnExp", 
               "Kn-FcCurviness", "Amp2ndMean", "Quality", "HiFtoFcExp", "LnExpA_EndAmp", "RelPwr2ndTo1st", "LnExpA_StartAmp", 
               "HiFminusStartF", "Amp3rdMean", "PreFc500Residue", "Kn-FcCurvinessTrndSlp", "PreFc250Residue", "AmpVariance", "AmpMoment", 
               "meanKn-FcCurviness", "MinAccpQuality", "AmpEndLn60ExpC", "AmpStartLn60ExpC", "Preemphasis", "MaxSegLnght" ,"Max#CallsConsidered" ]
ignore_cols += ["Filename", "NextDirUp", 'Path', 'Version', 'Filter', 'Preemphasis', 'MaxSegLnght', "ParentDir", "file_id", "chirp_idx", "split"]

In [8]:
data_module = stf.data.DataModule(
    datasetCls = BatsCSVDataset,
    dataset_kwargs = {
        "root_path": "../../bats_transformer/data/july_daytime_chunked_quantile/splits",
        "prefix": "split",
        "ignore_cols": ignore_cols,
        "time_col_name": "TimeIndex",
        "val_split": 0.05,
        "test_split": 0.05,
        "context_points": None,
        "target_points": 1,
        "random_seed": 31
    },
    batch_size=64,
    workers=4,
)

In [9]:
train_data = data_module.train_dataloader()
val_data = data_module.val_dataloader()
test_data = data_module.test_dataloader()

Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
Trying to unpickle estimator QuantileTransformer from version 1.3.2 when using version 1.3.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations


In [10]:
# preds = []
# for batch in tqdm(train_data):
#     x_t, x_c, y_t, y_c = batch
#     mask = x_t > 0
#     lengths = mask.sum(dim=1)
#     feature_sums = x_c.sum(dim=1)
#     # print(feature_sums.shape)
#     for i, row in enumerate(feature_sums):
#         preds.append((row / lengths[i]))
#     # print(preds)


In [11]:
truths = []
preds = []
errors = []

for batch in tqdm(test_data):
    x_t, x_c, y_t, y_c = batch
    mask = x_t > 0
    lengths = mask.sum(dim=1)
    feature_sums = x_c.sum(dim=1)
    for i, row in enumerate(feature_sums):
        pred = (row / lengths[i])
        truths.append(y_c[i].numpy()[0])
        preds.append(pred.numpy())
        errors.append((y_c[i] - pred).numpy()[0])

truths, preds, errors = np.array(truths), np.array(preds), np.array(errors)

100%|██████████| 22/22 [00:07<00:00,  3.12it/s]


In [12]:
pd.DataFrame(preds)

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,-0.215481,-0.196063,1.243677,1.307854,0.435711,-0.226535,0.768835,-0.546372,1.280677,0.835761,...,-0.192107,-0.203468,0.133628,0.102619,-0.352294,0.611969,0.589193,0.491467,0.288460,-0.159886
1,-0.311354,-0.155193,1.333033,1.378216,0.474706,-0.254928,0.957172,-0.824498,1.372436,0.840687,...,-0.161030,-0.155402,0.259783,0.019607,-0.243421,0.707019,0.635205,0.561974,0.300140,-0.221011
2,-0.418290,-0.217253,1.315596,1.395371,0.377578,-0.212951,0.876747,-0.780973,1.354674,0.811067,...,-0.168636,-0.205734,0.241700,0.110422,-0.290536,0.644137,0.545558,0.520450,0.223562,-0.142607
3,-0.532262,-0.357438,1.255686,1.329294,0.298631,-0.199725,0.788339,-0.769464,1.294139,0.787401,...,-0.233321,-0.192440,0.323503,0.040327,-0.372178,0.582643,0.483351,0.290390,-0.012423,-0.065570
4,-0.635543,-0.484763,1.174281,1.243227,0.243371,-0.222123,0.692798,-0.776818,1.210749,0.776363,...,-0.435784,-0.121554,0.365444,0.003957,-0.367738,0.472551,0.347794,0.207843,-0.132379,0.008215
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1377,-0.357396,-0.348907,1.061321,0.775509,0.223352,0.841818,0.532632,0.079358,1.099037,1.016919,...,0.424254,0.054123,0.154772,-0.221523,0.586365,0.600122,0.329198,0.598825,0.249694,0.556773
1378,-0.504821,-0.522424,1.253150,0.927079,0.252950,0.973599,0.618618,0.067286,1.292236,1.227056,...,0.381631,-0.108084,-0.062628,-0.427825,0.555483,0.598015,0.314463,0.572526,0.205384,0.580868
1379,-0.644045,-0.502132,1.380905,1.090662,0.229978,1.115660,0.597494,0.285170,1.424017,1.373291,...,0.582415,-0.123882,-0.146785,-0.504949,0.518763,0.628469,0.483189,0.649666,0.360028,0.868570
1380,-0.791541,-0.697346,1.412710,1.128369,0.175042,1.253690,0.651859,0.227572,1.455992,1.454857,...,0.540825,-0.136449,-0.148219,-0.612863,0.598165,0.614556,0.429795,0.664258,0.335564,1.089528


In [13]:
target_columns = train_data.dataset.target_cols
# target_columns

In [14]:
mae = np.abs(errors).mean(axis=0)
mse = (errors * errors).mean(axis=0)

In [15]:
mse

array([1.5915296 , 1.1999508 , 0.66019493, 0.8559924 , 0.48448893,
       0.98693585, 0.676982  , 0.86456513, 0.6800352 , 0.8377059 ,
       0.6849171 , 0.9157488 , 0.7295881 , 9.487489  , 0.84406966,
       0.9317536 , 0.5191961 , 0.6342059 , 0.6148738 , 0.6216553 ,
       0.4390783 , 0.53290343, 0.62657183, 0.8887792 , 0.80743074,
       0.79679173, 1.007295  , 0.8120128 , 1.1923922 , 0.72325236,
       1.1344643 , 0.9724536 ], dtype=float32)

In [16]:
mse_df = pd.DataFrame(np.array([target_columns, mse]).T)
mse_df = mse_df.set_index(0)
# mse_df[1] = mse_df[1].round(6)
pd.Series(mse, index=target_columns)

TimeInFile         1.591530
PrecedingIntrvl    1.199951
HiFreq             0.660195
Bndwdth            0.855992
FreqMaxPwr         0.484489
PrcntMaxAmpDur     0.986936
FreqKnee           0.676982
PrcntKneeDur       0.864565
StartF             0.680035
UpprKnFreq         0.837706
HiFtoUpprKnAmp     0.684917
HiFtoKnAmp         0.915749
HiFtoFcAmp         0.729588
UpprKnToKnAmp      9.487489
KnToFcAmp          0.844070
LdgToFcAmp         0.931754
FreqCtr            0.519196
FFwd32dB           0.634206
FFwd20dB           0.614874
FFwd15dB           0.621655
FBak5dB            0.439078
FFwd5dB            0.532903
Bndw32dB           0.626572
Amp1stQrtl         0.888779
Amp2ndQrtl         0.807431
Amp3rdQrtl         0.796792
Amp4thQrtl         1.007295
1st10kHzSlp        0.812013
1st5to15kHzSlp     1.192392
1st10kHzExp        0.723252
1st5to15kHzExp     1.134464
AmpK@start         0.972454
dtype: float32

In [17]:
average_loss_per_row = mse_df.mean(axis=1)
average_loss_per_row_no_outlier = mse_df.drop("UpprKnToKnAmp", axis=0).mean(axis=1)
print(average_loss_per_row.mean(), average_loss_per_row_no_outlier.mean())

1.0861032371875 0.8150907932258065
